In [ ]:
import google.generativeai as genai
import pandas as pd

genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-pro")

def diversify(tokens, labels):
    prompt = """
    You are a data generator for NER training.  
I will give you a sentence as tokens with BIO labels.  

Your job:
1. Rewrite the sentence into 3–5 diverse variations (resume, job description, LinkedIn post, HR email, etc.).  
2. Keep the same entity tokens intact (do not change or drop them).  
3. Return each variation in JSON format with:
   - "tokens": tokenized words of the new sentence
   - "labels": BIO labels aligned to each token

Example Input:
{
  "tokens": ["The", "ideal", "candidate", "should", "have", "completed", "GNM", "and", "possess", "strong", "woocommerce", "skills", "."],
  "labels": ["O","O","O","O","O","O","B-EDUCATION","O","O","O","B-SKILL","O","O"]
}

Example Output:
  {
    "tokens": ["Candidates", "must", "hold", "a", "GNM", "degree", "and", "show", "expertise", "in", "woocommerce", "."],
    "labels": ["O","O","O","O","B-EDUCATION","O","O","O","O","O","B-SKILL","O"]
  }
    """
    response = model.generate_content(prompt)
    return response.text

# Example with your dataset
row = {
  "tokens": ["The","ideal","candidate","should","have","completed","GNM","and","possess","strong","woocommerce","skills","."],
  "labels": ["O","O","O","O","O","O","B-EDUCATION","O","O","O","B-SKILL","O","O"]
}

print(diversify(row["tokens"], row["labels"]))


DefaultCredentialsError: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.

In [3]:
!pip install google-generativeai


  Using cached google_api_core-2.25.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached google_auth-2.40.3-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.70.0-py3-none-any.whl.metadata (9.3 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.73.0-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 16.9 MB/s eta 0:00:00
Using cached google_api_core-2.25.1-py3-none-any.whl (160 kB)
Using cached google_auth-2.40.3-py2.py3-none-any.whl (216 kB)
   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   ------------------- -------------------- 6.8/14.2 MB 32.3 MB/s eta 0:00:01
   --

In [ ]:
import google.generativeai as genai
import pandas as pd
import json
import time

# Configure your API key
genai.configure(api_key=)
model = genai.GenerativeModel("gemini-2.5-pro")

def diversify(tokens, labels):
    prompt = f"""
    You are a data generator for NER training.  
    I will give you a sentence as tokens with BIO labels.  

    Your job:
    1. Rewrite the sentence into 3–5 diverse variations (resume, job description, LinkedIn post, HR email, etc.).  
    2. Keep the same entity tokens intact (do not change or drop them).  
    3. Return each variation in JSON format with:
       - "tokens": tokenized words of the new sentence
       - "labels": BIO labels aligned to each token
    Input:
    {{
      "tokens": {tokens},
      "labels": {labels}
    }}
    """
    response = model.generate_content(prompt)
    # Try to parse multiple JSON objects if returned as a string list
    try:
        variations = json.loads(response.text)
        if isinstance(variations, dict):
            return [variations]
        elif isinstance(variations, list):
            return variations
    except json.JSONDecodeError:
        # fallback: return raw text
        return [{"tokens": tokens, "labels": labels}]
    return [{"tokens": tokens, "labels": labels}]

# File paths
input_file = r"C:\Users\gayat\DATA VISUALIZATION\data_visualization\random_indian_paragraphs_ner.jsonl"
output_file = r"C:\Users\gayat\DATA VISUALIZATION\data_visualization\ner_dataset_diversified.jsonl"

batch_size = 50  # process 50 rows at a time

with open(input_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
    batch = []
    for i, line in enumerate(f_in, 1):
        row = json.loads(line.strip())
        batch.append(row)

        # Process batch
        if i % batch_size == 0:
            for item in batch:
                tokens, labels = item["tokens"], item["labels"]
                variations = diversify(tokens, labels)
                for var in variations:
                    f_out.write(json.dumps(var, ensure_ascii=False) + "\n")
            print(f"Processed batch {i//batch_size} ({i} rows)")
            batch = []
            time.sleep(1)  # slight pause to avoid rate limits

    # Process remaining rows
    for item in batch:
        tokens, labels = item["tokens"], item["labels"]
        variations = diversify(tokens, labels)
        for var in variations:
            f_out.write(json.dumps(var, ensure_ascii=False) + "\n")
    if batch:
        print(f"Processed final batch ({len(batch)} rows)")

print(f"Diversified dataset saved to {output_file}")



ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50
Please retry in 7.041987215s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 50
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 7
}
]